# BMIN 5200 — Week 5 in-class exercise
## Evolving a solution: genetic algorithms and particle swarms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week05.ipynb)

**Time:** ~25 minutes · **Pairs with:** Biologically-inspired search (genetic algorithms, particle swarm optimization)

### What you'll do
- Reproduce the f(x) = x² genetic algorithm from the lecture slides, generation by generation, and watch average fitness climb
- Turn the same machinery loose on a real problem: choosing which of 12 candidate predictors belong in a 30-day readmission model
- Set the mutation rate to zero and watch the population go genetically uniform, which is how a GA fails in practice
- Run particle swarm optimization on a two-drug dose-response surface with a deceptive local optimum

### Why it matters
Feature selection, dose finding, and treatment-regimen optimization all have the same shape: a search space too large to enumerate, a fitness signal that is expensive to evaluate, and no gradient to follow. Evolutionary search is the standard tool when you can score a candidate but cannot differentiate the scoring function, which describes most clinical model-selection problems. TPOT, the automated pipeline builder you have probably seen in biomedical papers, is a genetic algorithm underneath, and it fails in exactly the way you will make this one fail.

## Setup

`scikit-learn`, `numpy`, and `matplotlib` are all preinstalled in Colab, so there is nothing to
install. The seed is fixed so that every laptop in the room shows the same generations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(5200)

## Part 1 — The GA from the slides, run for real

This is the toy from "Example: Optimize f(x) = x², 0 < x < 32". A chromosome is 5 bits, which
decodes to an integer 0–31, and fitness is that integer squared. The initial population is the
same four chromosomes as Generation 0 on the slide, so the numbers below should match the deck.
Nothing here is clinical yet; the point is to see every moving part of a GA in about twenty lines
before we aim it at something that matters.

In [ ]:
def decode(chromosome):
    """A chromosome is a list of 5 bits; the phenotype is the integer it spells in binary."""
    return int("".join(str(bit) for bit in chromosome), 2)


def fitness_squared(chromosome):
    return decode(chromosome) ** 2


def encode(value):
    return [int(bit) for bit in format(value, "05b")]


# Generation 0 from the lecture slide: 13, 24, 8, 19.
population = [encode(value) for value in (13, 24, 8, 19)]


def report_generation(generation_number, population):
    scores = [fitness_squared(c) for c in population]
    total = sum(scores)
    print(f"Generation {generation_number}")
    print(f"  {'chromosome':>12}  {'x':>3}  {'f(x) = x^2':>10}  {'share of wheel':>14}")
    for chromosome, score in zip(population, scores):
        bits = "".join(str(bit) for bit in chromosome)
        print(f"  {bits:>12}  {decode(chromosome):>3}  {score:>10}  {score / total:>13.1%}")
    print(f"  sum = {total}    average = {total / len(scores):.1f}    max = {max(scores)}")
    print()


report_generation(0, population)

Those four fitness values sum to 1170 and average 292.5, which is what the slide says. The
"share of wheel" column is the roulette wheel from "How do you select chromosomes for
reproduction or deletion?": chromosome `11000` occupies about half of it, so it is expected to be
selected roughly twice. Below are the three operators from "Genetic Operators" — selection,
one-point crossover, bit-flip mutation — each written as a plain function so you can see there is
no magic in any of them.

In [ ]:
def roulette_selection(population, fitness_fn, rng):
    """Sample len(population) parents with probability proportional to fitness."""
    scores = np.array([fitness_fn(c) for c in population], dtype=float)
    if scores.sum() == 0:
        probabilities = np.ones(len(population)) / len(population)
    else:
        probabilities = scores / scores.sum()
    chosen = rng.choice(len(population), size=len(population), p=probabilities)
    return [list(population[i]) for i in chosen]


def one_point_crossover(parent_a, parent_b, rng):
    """Cut both parents at the same random point and swap the tails."""
    cut = int(rng.integers(1, len(parent_a)))
    return parent_a[:cut] + parent_b[cut:], parent_b[:cut] + parent_a[cut:]


def bit_flip_mutation(chromosome, mutation_rate, rng):
    # Mutation is the only operator that can introduce an allele the population has never held.
    return [1 - bit if rng.random() < mutation_rate else bit for bit in chromosome]


def next_generation(population, fitness_fn, mutation_rate, rng):
    parents = roulette_selection(population, fitness_fn, rng)
    children = []
    for i in range(0, len(parents), 2):
        child_a, child_b = one_point_crossover(parents[i], parents[i + 1], rng)
        children.append(bit_flip_mutation(child_a, mutation_rate, rng))
        children.append(bit_flip_mutation(child_b, mutation_rate, rng))
    return children


population = [encode(value) for value in (13, 24, 8, 19)]
for generation in range(1, 5):
    population = next_generation(population, fitness_squared, 0.02, rng)
    report_generation(generation, population)

Read the `average` line down the four generations: it rises steadily, and by Generation 4 the
population sits near the top of the range. Notice what the GA was never told — that f(x) = x² is
increasing in x, that the answer is 31, or what a derivative is. It only ever asked "how good is
this one?" and let selection do the rest. That is the whole proposition of evolutionary search,
and it is why it works on objectives you cannot differentiate.

## Part 2 — A GA that picks predictors for 30-day readmission

Same three operators, real problem. We have 12 candidate predictors of 30-day readmission and a
synthetic cohort of 400 patients. The data are generated below, so we know the ground truth: only
four of the twelve carry any signal. A chromosome is now 12 bits, where bit *i* is 1 if predictor
*i* is in the model, and fitness is cross-validated AUC minus a penalty for each predictor kept.
The penalty is how you tell a genetic algorithm that you prefer a parsimonious model.

In [ ]:
# Synthetic cohort. These are not real patients; the generative model is fully known to us.
CANDIDATE_PREDICTORS = [
    "age", "prior_admissions", "hemoglobin", "sodium", "creatinine", "ejection_fraction",
    "bmi", "systolic_bp", "hba1c", "discharge_meds", "los_days", "albumin",
]
TRULY_PREDICTIVE = ["prior_admissions", "creatinine", "ejection_fraction", "discharge_meds"]

n_patients = 400
predictors = rng.normal(size=(n_patients, len(CANDIDATE_PREDICTORS)))

true_weights = np.zeros(len(CANDIDATE_PREDICTORS))
for name, weight in zip(TRULY_PREDICTIVE, [1.1, 1.0, -1.2, 0.9]):
    true_weights[CANDIDATE_PREDICTORS.index(name)] = weight

log_odds = predictors @ true_weights - 0.3
readmitted = (rng.random(n_patients) < 1 / (1 + np.exp(-log_odds))).astype(int)

print(f"{n_patients} synthetic patients, {readmitted.mean():.1%} readmitted within 30 days")
print(f"candidate predictors: {len(CANDIDATE_PREDICTORS)}")
print(f"subsets a GA could be asked to consider: 2^12 = {2 ** 12}")
print(f"predictors that actually carry signal: {', '.join(TRULY_PREDICTIVE)}")

2¹² is only 4096, which is deliberate: it is small enough that we can check the GA's answer
against the truth, and against brute force if we want to. Add ten more candidate predictors and
2²² is four million model fits, which is where enumeration dies.

Your first job is the fitness function. Cross-validated AUC on its own will happily keep noise
predictors, because on 400 patients a spurious variable barely moves AUC at all. The penalty term
is what makes four predictors beat five.

In [ ]:
fitness_cache = {}  # Refitting the same subset is the expensive part, so memoize it.


def subset_fitness(chromosome, penalty_per_predictor=0.01):
    key = tuple(chromosome)
    if key in fitness_cache:
        return fitness_cache[key]

    columns = [i for i, bit in enumerate(chromosome) if bit == 1]
    if len(columns) == 0:
        fitness_cache[key] = 0.0          # An empty model has no discrimination at all.
        return 0.0

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    auc = cross_val_score(model, predictors[:, columns], readmitted, cv=3, scoring="roc_auc").mean()

    # TODO: subtract penalty_per_predictor for EACH predictor this chromosome keeps, so that a
    #       four-predictor model beats a five-predictor model of the same AUC. Replace this line.
    score = auc

    fitness_cache[key] = score
    return score


def as_chromosome(names):
    return [1 if predictor in names else 0 for predictor in CANDIDATE_PREDICTORS]


true_four = as_chromosome(TRULY_PREDICTIVE)
four_plus_noise = as_chromosome(TRULY_PREDICTIVE + ["sodium"])

print(f"fitness of the true 4 predictors:        {subset_fitness(true_four):.4f}")
print(f"fitness of those 4 plus sodium (noise):  {subset_fitness(four_plus_noise):.4f}")
print(f"margin in favour of the smaller model:   {subset_fitness(true_four) - subset_fitness(four_plus_noise):+.4f}")
print()
print("With the placeholder (no penalty) that margin is a rounding error and can even come out")
print("negative: at n=400, cross-validated AUC cannot separate a useful predictor from a useless")
print("one. Once you subtract 0.01 per predictor the margin is about +0.008, which is a")
print("difference selection can act on.")

Fill in the penalty above and rerun that cell before continuing. Now the GA itself. Selection
changes from roulette to a three-way tournament, because roulette needs strictly positive fitness
and does badly when every candidate scores within a few percent of every other, which is exactly
our situation. We also carry the single best chromosome unchanged into the next generation
("elitism") so a good solution cannot be lost to an unlucky draw.

In [ ]:
def tournament_selection(population, scores, rng, k=3):
    """Pick k chromosomes at random and return a copy of the fittest of them."""
    contenders = rng.integers(0, len(population), size=k)
    winner = max(contenders, key=lambda i: scores[i])
    return list(population[winner])


def run_feature_ga(population_size=16, n_generations=12, mutation_rate=0.03, seed=5200):
    """One row per generation: (generation, best fitness, distinct chromosomes, best chromosome)."""
    ga_rng = np.random.default_rng(seed)
    population = [list(bits) for bits in ga_rng.integers(0, 2, size=(population_size, 12))]

    history = []
    for generation in range(n_generations + 1):
        scores = [subset_fitness(c) for c in population]
        best = int(np.argmax(scores))
        distinct = len({tuple(c) for c in population})
        history.append((generation, scores[best], distinct, list(population[best])))
        if generation == n_generations:
            break

        children = [list(population[best])]        # elitism
        while len(children) < population_size:
            parent_a = tournament_selection(population, scores, ga_rng)
            parent_b = tournament_selection(population, scores, ga_rng)
            child_a, child_b = one_point_crossover(parent_a, parent_b, ga_rng)
            children.append(bit_flip_mutation(child_a, mutation_rate, ga_rng))
            children.append(bit_flip_mutation(child_b, mutation_rate, ga_rng))
        population = children[:population_size]
    return history


def report_run(history, title):
    print(title)
    print(f"  {'gen':>3}  {'best fitness':>12}  {'distinct':>8}  predictors in the best chromosome")
    for generation, best_score, distinct, chromosome in history:
        kept = [CANDIDATE_PREDICTORS[i] for i, bit in enumerate(chromosome) if bit == 1]
        print(f"  {generation:>3}  {best_score:>12.4f}  {distinct:>8}  {', '.join(kept)}")
    print()


report_run(run_feature_ga(), "population 16, mutation rate 0.03")
print(f"ground truth: {', '.join(TRULY_PREDICTIVE)}")

With the penalty in place the GA lands on exactly the four predictors that generated the data,
having scored a couple of hundred of the 4096 possible subsets. Watch the `distinct` column too.
It starts at 16 — every chromosome different — and drifts down as the population fills with
descendants of whatever was working at the time. That drift is the GA's real vulnerability, and
it is what we break next.

### Predict before you run

We are about to rerun the identical GA — same seed, same starting population, same tournament,
same crossover — with `mutation_rate = 0.0`. Crossover on its own can still shuffle predictors
between chromosomes, so the search is not obviously dead.

Commit to an answer out loud before running the next cell:

1. Does it still find the true four predictors?
2. What happens to the `distinct` column by generation 12?
3. Once every chromosome in the population agrees on some bit, is there anything left that can
   change it?

In [ ]:
with_mutation = run_feature_ga(mutation_rate=0.03)
without_mutation = run_feature_ga(mutation_rate=0.0)

report_run(with_mutation, "mutation rate 0.03  (the run from above)")
report_run(without_mutation, "mutation rate 0.00  (nothing else changed)")

final_chromosome = without_mutation[-1][3]
kept = [CANDIDATE_PREDICTORS[i] for i, bit in enumerate(final_chromosome) if bit == 1]
spurious = [predictor for predictor in kept if predictor not in TRULY_PREDICTIVE]

print(f"final fitness with mutation:    {with_mutation[-1][1]:.4f}")
print(f"final fitness without mutation: {without_mutation[-1][1]:.4f}")
print(f"distinct chromosomes left in the mutation-free population: {without_mutation[-1][2]}")
print(f"noise predictors it could never drop: {', '.join(spurious) if spurious else 'none'}")

By generation 10 the mutation-free population holds **one** distinct chromosome, repeated
sixteen times. Crossover between two identical parents returns those same parents, so nothing new
can ever be produced again; the search has stopped and is just burning generations. It also
stopped on the wrong answer: the last line of output names the noise predictors it could never
drop, and it could never drop them because at the moment diversity ran out, no chromosome in the
pool carried a 0 in those positions. This is premature convergence. When an AutoML run plateaus
for fifty generations, this is usually what happened.

## Part 3 — Particle swarm optimization on a dose-response surface

PSO keeps the population idea and drops chromosomes entirely. A particle is a *position* in the
search space plus a *velocity*, and each iteration pulls it toward two attractors: the best point
that particle has personally seen (`personal_best`, the cognitive term) and the best point any
particle has seen (`global_best`, the social term). Our search space is a two-drug dose
combination in mg/kg and the surface is tumor response, with a broad mediocre optimum near
(2.5, 2.0) and a narrow, much better one near (8.0, 7.0). The easy basin is far larger, which is
what makes the surface deceptive.

In [ ]:
def tumor_response(dose_a, dose_b):
    """Synthetic two-drug response surface, doses in mg/kg. Two peaks; the better one is narrow."""
    broad_mediocre = 0.95 * np.exp(-((dose_a - 2.5) ** 2 + (dose_b - 2.0) ** 2) / 6.0)
    narrow_best = 1.60 * np.exp(-((dose_a - 8.0) ** 2 + (dose_b - 7.0) ** 2) / 1.2)
    return broad_mediocre + narrow_best


def run_pso(n_particles=25, n_iterations=30, inertia=0.72, cognitive=1.4, social=1.4,
            seed=5200, snapshot_at=(0, 3, 10, 30)):
    swarm_rng = np.random.default_rng(seed)
    position = swarm_rng.uniform(0, 10, size=(n_particles, 2))
    velocity = swarm_rng.uniform(-1, 1, size=(n_particles, 2))

    personal_best = position.copy()
    personal_best_value = tumor_response(position[:, 0], position[:, 1])
    leader = int(np.argmax(personal_best_value))
    global_best = personal_best[leader].copy()
    global_best_value = personal_best_value[leader]

    snapshots = {0: position.copy()}
    for iteration in range(1, n_iterations + 1):
        r_cognitive = swarm_rng.random((n_particles, 2))
        r_social = swarm_rng.random((n_particles, 2))

        # TODO: this is the velocity update from the "Pseudocode for PSO" slide with the social
        #       term missing, so each particle only chases its own personal best and the 25
        #       particles never become a swarm. Add the third term:
        #           + social * r_social * (global_best - position)
        velocity = inertia * velocity + cognitive * r_cognitive * (personal_best - position)

        position = np.clip(position + velocity, 0, 10)   # doses cannot go below 0 or above 10 mg/kg

        value = tumor_response(position[:, 0], position[:, 1])
        improved = value > personal_best_value
        personal_best[improved] = position[improved]
        personal_best_value[improved] = value[improved]

        leader = int(np.argmax(personal_best_value))
        if personal_best_value[leader] > global_best_value:
            global_best = personal_best[leader].copy()
            global_best_value = personal_best_value[leader]

        if iteration in snapshot_at:
            snapshots[iteration] = position.copy()

    return snapshots, global_best, global_best_value


snapshots, best_dose, best_value = run_pso()
print(f"best combination found:  drug A {best_dose[0]:.2f} mg/kg, drug B {best_dose[1]:.2f} mg/kg"
      f"  ->  response {best_value:.3f}")
print("true global optimum:     drug A 8.00 mg/kg, drug B 7.00 mg/kg  ->  response 1.600")
print("deceptive local optimum: drug A 2.50 mg/kg, drug B 2.00 mg/kg  ->  response 0.950")

Now plot where the swarm actually is at four moments. With the social term missing you get 25
particles milling around independently and staying spread out; with it in, they visibly collapse
onto one basin between iteration 3 and iteration 30. The contour lines are the response surface,
the star is the global optimum, and the square is the deceptive one.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(9, 8))
grid_a, grid_b = np.meshgrid(np.linspace(0, 10, 120), np.linspace(0, 10, 120))
surface = tumor_response(grid_a, grid_b)

for axis, iteration in zip(axes.flat, sorted(snapshots)):
    axis.contour(grid_a, grid_b, surface, levels=12)
    swarm = snapshots[iteration]
    axis.plot(swarm[:, 0], swarm[:, 1], "o", markersize=5)
    axis.plot(8.0, 7.0, "*", markersize=16, color="white", markeredgecolor="black")
    axis.plot(2.5, 2.0, "s", markersize=8, color="white", markeredgecolor="black")
    axis.set_title(f"iteration {iteration}  (spread = {swarm.std():.2f})")
    axis.set_xlabel("drug A (mg/kg)")
    axis.set_ylabel("drug B (mg/kg)")
    axis.set_xlim(0, 10)
    axis.set_ylim(0, 10)

figure.suptitle("Swarm positions: star = global optimum, square = deceptive local optimum")
figure.tight_layout()
plt.show()

The last experiment is the honest one. Nothing about PSO guarantees it finds the global
optimum; it guarantees only that the swarm eventually agrees on something. The cell below runs the
same swarm from three different random initializations, with the social pull off and then on. Off,
the particles never converge but between them they cover the space. On, they converge tightly —
and on two of the three seeds they converge on the wrong peak, because once `global_best` sits at
(2.5, 2.0) every particle is dragged toward it and the narrow peak is never sampled again.

In [ ]:
print(f"  {'social':>7}  {'seed':>5}  {'drug A':>7}  {'drug B':>7}  {'response':>9}  {'spread':>7}  outcome")
for social_weight in (0.0, 1.4):
    for seed in (5200, 1, 7):
        snaps, dose, value = run_pso(social=social_weight, seed=seed)
        spread = snaps[30].std()
        outcome = "reached the global optimum" if value > 1.2 else "stuck on the local optimum"
        print(f"  {social_weight:>7.1f}  {seed:>5}  {dose[0]:>7.2f}  {dose[1]:>7.2f}"
              f"  {value:>9.3f}  {spread:>7.2f}  {outcome}")

print()
print("If the two blocks are identical, the social term is not in your velocity update yet.")
print("Spread is the standard deviation of particle coordinates at iteration 30: small means")
print("the swarm has committed, which is exactly what makes it possible to commit to the wrong peak.")

## Talk about it

1. The mutation-free GA returned a model containing `sodium` and `systolic_bp`, two variables
   with no relationship to readmission at all, and reported a respectable AUC while doing it. If
   you had not generated the data yourself, is there anything in the GA's own output that would
   have warned you?

2. Our fitness was cross-validated AUC minus 0.01 per predictor. Where did 0.01 come from? Try
   0.05 and 0.001 and watch the answer change. If a free parameter you chose determines which
   predictors appear in your published model, what has the automation actually bought you?

3. Turning the social term on made PSO converge, and made it converge on the wrong dose two times
   out of three. In a dose-finding study you get one run, you cannot see the surface, and the
   swarm looks equally confident either way. What would you need to add before this could inform
   a real dose-escalation protocol?

## Solutions

Completed versions of the TODOs above. These are markdown, not code cells, so scrolling down here
will not overwrite your own work.

**Part 2 — the parsimony penalty**

```python
    # One hundredth of an AUC point per predictor: a variable has to earn its place.
    score = auc - penalty_per_predictor * len(columns)
```

**Part 3 — the full PSO velocity update**

```python
        velocity = (inertia * velocity
                    + cognitive * r_cognitive * (personal_best - position)
                    + social * r_social * (global_best - position))
```

**Optional — check the GA against brute force.** All 4096 subsets, roughly 20 seconds. This is
only tractable because we deliberately kept the problem to 12 candidate predictors.

```python
import itertools

best_subset, best_score = None, -1.0
for bits in itertools.product([0, 1], repeat=12):
    score = subset_fitness(list(bits))
    if score > best_score:
        best_subset, best_score = list(bits), score

print([CANDIDATE_PREDICTORS[i] for i, bit in enumerate(best_subset) if bit], round(best_score, 4))
```
